# Fine-Tune Whisper For Multilingual ASR with Transformers





In [2]:
!jupyter kernelspec list

Available kernels:
  python3     /home/orgpu/copied/voicepro_training/ft-whisper/weesper/share/jupyter/kernels/python3
  _venv       /home/orgpu/.local/share/jupyter/kernels/_venv
  _venv_vp    /home/orgpu/.local/share/jupyter/kernels/_venv_vp
  ft          /home/orgpu/.local/share/jupyter/kernels/ft
  weesper     /home/orgpu/.local/share/jupyter/kernels/weesper


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Wed Jan 14 13:13:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 NVL                Off |   00000000:00:10.0 Off |                    0 |
| N/A   41C    P0             60W /  350W |      14MiB /  95830MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
%pip install --upgrade pip
%pip install --upgrade datasets[audio] transformers accelerate evaluate jiwer tensorboard
%pip install soundfile librosa


Note: you may need to restart the kernel to use updated packages.
  Using cached transformers-4.57.5-py3-none-any.whl.metadata (43 kB)
  Using cached torchcodec-0.9.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylin

In [4]:
# 1. Install PyTorch compatible with H100/CUDA 12
%pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

# 2. Upgrade transformers and datasets
%pip install -U transformers datasets

# 3. Install sentencepiece for Whisper
%pip install sentencepiece







Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl (664.8 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached https://download.pytorch.org/wh

In [5]:
%pip uninstall transformers -y
%pip install transformers==4.57.1



Found existing installation: transformers 4.57.5
Uninstalling transformers-4.57.5:
  Successfully uninstalled transformers-4.57.5
Note: you may need to restart the kernel to use updated packages.
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install -U datasets


Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip uninstall -y torchcodec datasets
%pip install datasets transformers==4.57.1
%pip install soundfile librosa

Found existing installation: torchcodec 0.9.1
Uninstalling torchcodec-0.9.1:
  Successfully uninstalled torchcodec-0.9.1
Found existing installation: datasets 4.4.2
Uninstalling datasets-4.4.2:
  Successfully uninstalled datasets-4.4.2
Note: you may need to restart the kernel to use updated packages.
  Using cached datasets-4.4.2-py3-none-any.whl.metadata (19 kB)
Using cached datasets-4.4.2-py3-none-any.whl (512 kB)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Setting Model

In [3]:
lm = "openai/whisper-large-v2"
ver = "_mix_ur_ps_pn_v0.3"
language = "ur"
# task = "transcribe"

## Load Dataset

In [4]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files={"train": "urdu_pashtu_punjabi_train_v0.3.csv", 
                                          "val": "urdu_pashtu_punjabi_val_v0.3.csv", 
                                          "test": "urdu_pashtu_punjabi_test_v0.3.csv"})

dataset

/home/orgpu/copied/voicepro_training/ft-whisper/weesper/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['path', 'language', 'task', 'text'],
        num_rows: 189646
    })
    val: Dataset({
        features: ['path', 'language', 'task', 'text'],
        num_rows: 10536
    })
    test: Dataset({
        features: ['path', 'language', 'task', 'text'],
        num_rows: 10536
    })
})

In [5]:
from datasets import Value

train_set = dataset["train"].cast_column("path", Value("string"))
val_set = dataset["val"].cast_column("path", Value("string"))
test_set = dataset["test"].cast_column("path", Value("string"))

In [6]:
train_set

Dataset({
    features: ['path', 'language', 'task', 'text'],
    num_rows: 189646
})

In [12]:
# %pip install -U "transformers>=4.46.2" "huggingface-hub>=0.25,<1.0" tokenizers
# train_set

### Load WhisperFeatureExtractor

In [7]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer

feature_extractor = WhisperFeatureExtractor.from_pretrained(lm)

In [8]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained(lm)
tokenizer = WhisperTokenizer.from_pretrained(lm, language="urdu", task="transcribe")
processor = WhisperProcessor.from_pretrained(lm, language="urdu", task="transcribe")

In [9]:
from transformers import WhisperProcessor
import soundfile as sf

# Load processor once
processor = WhisperProcessor.from_pretrained(lm)  # lm = your model, e.g., "openai/whisper-small"

# Language and task mappings
LANG_MAPPING = {
    "urdu": "ur",
    "pashtu": "ps",
    "pashto": "ps",
    "punjabi": "pa"
}
TASK_MAPPING = {
    "transcription": "transcribe",
    "translation": "translate"
}

def prepare_dataset(batch):
    """
    Converts audio paths to input_features and text to tokenized labels.
    Skips corrupted audio files safely.
    """
    audio_path = batch["path"]

    # --- AUDIO FEATURE EXTRACTION ---
    try:
        audio, sr = sf.read(audio_path)
    except Exception as e:
        print(f"Skipping corrupted file: {audio_path} | Error: {e}")
        return None  # datasets.map will ignore None results if batched=False

    batch["input_features"] = processor.feature_extractor(
        audio, sampling_rate=16000
    ).input_features[0]

    # --- LANGUAGE & TASK MAPPING ---
    lang_code = LANG_MAPPING.get(batch["language"].lower(), "en")  # default to "en"
    task_name = TASK_MAPPING.get(batch["task"].lower(), "transcribe")

    batch["forced_decoder_ids"] = processor.tokenizer.get_decoder_prompt_ids(
        language=lang_code,
        task=task_name
    )

    # --- TOKENIZE LABELS ---
    # Whisper uses text_target for labels
    batch["labels"] = processor.tokenizer(
        text=batch["text"],
        text_target=batch["text"],  # labels
        add_special_tokens=True
    ).input_ids

    return batch

In [10]:
dataset_train = train_set.map(
    prepare_dataset,
    remove_columns=train_set.column_names,
    num_proc=4
).filter(lambda x: x is not None)

In [11]:
print(
    processor.tokenizer.decode(
        dataset_train[8]["labels"],
        skip_special_tokens=False
    )
)

<|startoftranscript|><|ur|><|transcribe|><|notimestamps|>انہیں آگے بڑھانا اور ترقیاں دینا، انہیں جتنی تیری ذات سے امیدیں ہیں اس سے زیادہ ان کی مدد فرمانا۔ ان کی زندگیوں میں اجالے بھرنا، نصیب اچھے کرنا اور ہر بے اولاد کو اولاد دینا۔ آمین ثم آمین۔<|endoftext|>


In [12]:
dataset_val = val_set.map(
    prepare_dataset,
    remove_columns=val_set.column_names,
    num_proc=4
).filter(lambda x: x is not None)

### Prepare Data

Let's print the example from our dataset to see
what form the data is in:

Re-loading the first audio sample in the Common Voice dataset will resample
it to the desired sampling rate:

In [13]:

# print("------------------------")
print(dataset_train[0])

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



### Define a Data Collator

In [14]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [15]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
# data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor, decoder_start_token_id=model.config.decoder_start_token_id)

Let's initialise the data collator we've just defined:

### Evaluation Metrics

In [23]:
from transformers import WhisperForConditionalGeneration
model = WhisperForConditionalGeneration.from_pretrained(lm)

### Define the Training Configuration

In the final step, we define all the parameters related to training. For more detail on the training arguments, refer to the Seq2SeqTrainingArguments [docs](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Seq2SeqTrainingArguments).

In [24]:
from transformers import Seq2SeqTrainingArguments, EarlyStoppingCallback

training_args = Seq2SeqTrainingArguments(
    output_dir=f"./{lm+ver}_urdu_pashtu_panjabi",  # change to a repo name of your choice
    per_device_train_batch_size=10,
    gradient_accumulation_steps=4,  # increase by 2x for every 2x decrease in batch size
    num_train_epochs=12,
    eval_strategy="epoch",
    save_strategy="epoch",        # saves at the end of each epoch
    logging_strategy="epoch",
    learning_rate=1e-4,
    predict_with_generate=False,
    generation_max_length=448,
    fp16=True,
    load_best_model_at_end=True,
    # metric_for_best_model="wer",
    # greater_is_better=False,
    report_to=["tensorboard"],
    remove_unused_columns=False,
    label_names=["labels"],  # same reason as above
)

In [25]:
model.config.use_cache = False

In [26]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    data_collator=data_collator,
    # compute_metrics=compute_metrics,
    tokenizer=processor.tokenizer,
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

/tmp/ipykernel_1924896/1912966190.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


### Training

In [30]:
trainer.train()
# trainer.train(resume_from_checkpoint="openai/whisper-large-v2_mix_ur_ps_pn_v0.3_urdu_pashtu_panjabi/checkpoint-4737")


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
model.save_pretrained(f"./{lm+ver}")
tokenizer.save_pretrained(f"./{lm+ver}")  # Optional but recommended
processor.save_pretrained(f"./{lm+ver}")  # Optional but recommended
